In [1]:
%pip install -q pandas numpy matplotlib scikit-learn
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Note: you may need to restart the kernel to use updated packages.


Rows: 10879 | Years: 1791 to 2026


# 01. The complete dataset

## What this notebook does
It builds data/processed/executive_orders.csv from the two raw sources, using scripts/build_processed.py, then shows the dataset the way a beginner would want to see it: how many orders, for which Presidents, which parties, which eras, and which terms.

## How the build works, in plain words
APP gives one row per listed order. The Federal Register gives official numbers, dates, titles, and disposition notes for recent orders. When both sources have the same EO number, they are merged into one row, and the Federal Register title and date win, because it is the official journal. When only one source has a row, that row is kept. Nothing is invented. If a title is just the words Executive Order with no subject, it stays that way, and it is counted as a generic title in notebook 02.

Run the build cell now. It prints a short audit trail.

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, str(BASE / "scripts" / "build_processed.py")], capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode != 0 else "Build script exit code: 0")

Dropped 1 exact duplicate rows with an EO number (APP listing duplicates, for example EO 8382 listed twice). Same day generic titles with no number are kept, because they may be distinct orders.
APP rows: 10880
FR unique EO numbers: 1549
FR rows enriched into APP rows: 1549
FR rows added as new rows: 0
Final dataset rows: 10879
Date range: 1791-01-22 to 2026-10-22
Rows with EO number: 9989
Rows with full text: 1549
Wrote /home/hatch/workspace/us-executive-orders/data/processed/executive_orders.csv

Build script exit code: 0


In [3]:
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Final rows:", len(DF))
print(DF.head(3).to_string())
print()
print(DF.columns.tolist())

Final rows: 10879
   executive_order_number          president                  party                           term           era date_signed  year  decade                                                                                                                                        title                                                                                                                                  title_clean  title_word_count               category category_keyword disposition_status disposition_notes                       source text_available  text_word_count                                                                                                                               app_url federal_register_url president_app_label
0                     NaN  George Washington            Independent  George Washington (1789-1797)  Founding era  1791-01-22  1791    1790  Commission - Appointing Commissioners for Surveying the District of Territory for the Perm

## Columns, explained
executive_order_number: the number if the order has one, blank for unnumbered early orders. president, party, term, era: assigned from the signing date using the term table in scripts/presidents.py. date_signed, year, decade: from the source date. title: the source title. title_clean: the title with the leading Executive Order number removed, so word counts are not swamped by the word Executive. title_word_count: words in the clean title. category, category_keyword: the keyword rule result from notebook 03. disposition_status, disposition_notes: what the Federal Register notes say about revocation or amendment, or Unknown when no notes exist. source: which source or sources supplied the row. text_available, text_word_count: whether a full text file was downloaded here, and how many words it has. app_url, federal_register_url: links back to the sources.

In [4]:
by_pres = DF.groupby(["president", "party"]).size().reset_index(name="orders")
by_pres = by_pres.sort_values("orders", ascending=False)
print(by_pres.to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 6))
top = by_pres.head(12).iloc[::-1]
ax.barh(top["president"], top["orders"], color="#5b7fa6")
ax.set_title("Orders in the dataset, top 12 Presidents")
ax.set_xlabel("Orders in dataset rows")
plt.tight_layout()
plt.savefig(CH / "01_orders_by_president.png")
plt.show()

            president                 party  orders
Franklin D. Roosevelt            Democratic    3725
       Herbert Hoover            Republican    1004
      Harry S. Truman            Democratic     907
      Donald J. Trump            Republican     512
 Dwight D. Eisenhower            Republican     484
        Ronald Reagan            Republican     381
   William J. Clinton            Democratic     364
        Richard Nixon            Republican     346
    Lyndon B. Johnson            Democratic     325
         Jimmy Carter            Democratic     320
       George W. Bush            Republican     291
         Barack Obama            Democratic     276
      John F. Kennedy            Democratic     214
     Grover Cleveland            Democratic     173
          Gerald Ford            Republican     169
    George H. W. Bush            Republican     166
      Joseph R. Biden            Democratic     162
       Woodrow Wilson            Democratic     150
      Abraha

In [5]:
print("By party:")
print(DF.groupby("party").size().to_string())
print()
print("By era:")
print(DF.groupby("era").size().to_string())
print()
print("By term, recent terms:")
recent = DF[DF["year"] >= 1989].groupby("term").size()
print(recent.to_string())
print()
print("Generic titles (title is only Executive Order, no subject):")
gen = DF[DF["title_clean"].str.strip() == ""]
print("Generic title rows:", len(gen), "| Share:", round(100 * len(gen) / len(DF), 1), "percent")
print(gen.groupby("president").size().sort_values(ascending=False).head(8).to_string())

By party:
party
Democratic               6702
Democratic-Republican       2
Independent                 1
Republican               4164
Whig                       10

By era:
era
Antebellum era                    26
Biden era                        163
Civil War and Reconstruction     315
Founding era                       3
Gilded Age                       423
Interwar years                  1022
New Deal and World War II       3952
Obama era                        274
Post Cold War                    358
Postwar and Cold War            1782
Progressive Era                  360
Reagan era and Cold War end      567
Trump first term                 213
Trump second term                305
Vietnam and Watergate era        818
War on Terror era                298

By term, recent terms:
term
Barack Obama (2009-2017)          276
Donald J. Trump (2017-2021)       220
Donald J. Trump (2025-2029)       292
George H. W. Bush (1989-1993)     166
George W. Bush (2001-2009)        291
Joseph R. 

In [6]:
DF.to_csv(BASE / "data" / "processed" / "executive_orders.csv", index=False)
print("Saved:", BASE / "data" / "processed" / "executive_orders.csv", "| Rows:", len(DF))

Saved: /home/hatch/workspace/us-executive-orders/data/processed/executive_orders.csv | Rows: 10879
